# GRPO Training — Clinical Data Cross-Domain Validation (Task 4)

This notebook fine-tunes a 7B LLM using GRPO (Group Relative Policy Optimization) on Task 4 of the Clinical Data Standardization environment.

**Baseline (zero-shot llama-3.3-70b-versatile):** 0.4388  
**Target:** 0.65+ on the pharmaverse Task 4 benchmark

**Hardware required:** Kaggle P100 or T4 (16GB VRAM)

## Cell 1 — Install dependencies

In [1]:
%%capture
!pip install -q trl>=0.12.0 transformers>=4.47.0 peft>=0.14.0 bitsandbytes>=0.45.0 accelerate>=1.2.0 datasets huggingface_hub


## Cell 2 — Clone repo and import grader

In [2]:
import subprocess, sys, json, os

# Clone the environment repo
if not os.path.exists('/kaggle/working/clinical-data-env'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/thisis-gp/clinical-data-env.git',
         '/kaggle/working/clinical-data-env'],
        check=True
    )

# Add grader to path
REPO = '/kaggle/working/clinical-data-env'
sys.path.insert(0, REPO)

from graders.grader_task4 import grade_task4

print('Grader imported successfully')

Grader imported successfully


## Cell 3 — Load Task 4 cases and build dataset

In [3]:
import json
from datasets import Dataset

SYSTEM_PROMPT = """You are a clinical data standardization expert trained in CDISC standards.

You will receive cross-domain SDTM records (DM, EX, CM, AE, DS) and must identify any inconsistencies.
You must respond with ONLY a valid JSON object — no markdown, no explanation outside the JSON.

Your response must have this exact structure:
{"issues": [{"type": "...", "domain": "...", "field": "...", "description": "..."}]}

If no cross-domain inconsistencies exist, return: {"issues": []}

Use these canonical issue types:
  - "dm_ex_date_mismatch" — DM.RFSTDTC does not match earliest EX.EXSTDTC
  - "prohibited_cm_before_first_dose" — prohibited CM starts before RFSTDTC
  - "orphan_sae" — AE with AESER='Y' has no matching DS record

Return ONLY the JSON object."""


def load_cases(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)['cases']


def case_to_prompt(case):
    """Convert a task4 case to a chat prompt."""
    user_content = (
        f"STUDY CONTEXT & RULES:\n{case['study_context']}\n\n"
        f"TASK:\n{case['task_description']}\n\n"
        f"INPUT DATA:\n{json.dumps(case['input_data'], indent=2)}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


# Load toy + pharmaverse cases for training
toy_cases = load_cases(f'{REPO}/data/task4_cases.json')
pharmaverse_cases = load_cases(f'{REPO}/data/pharmaverse/task4_cases.json')
all_cases = toy_cases + pharmaverse_cases

print(f'Total training cases: {len(all_cases)} ({len(toy_cases)} toy + {len(pharmaverse_cases)} pharmaverse)')

# Build HuggingFace dataset — store ground truth in extra column for reward fn
records = [
    {
        'prompt': case_to_prompt(case),
        'ground_truth': json.dumps(case['ground_truth']),
        'case_id': case['case_id'],
        'difficulty': case.get('difficulty', 'unknown'),
    }
    for case in all_cases
]

dataset = Dataset.from_list(records)
print(dataset)

Total training cases: 20 (10 toy + 10 pharmaverse)
Dataset({
    features: ['prompt', 'ground_truth', 'case_id', 'difficulty'],
    num_rows: 20
})


## Cell 4 — Define reward function

In [4]:
import json
import re


def normalize_completion_text(completion) -> str:
    """Convert TRL completion payloads into plain text."""
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                if "content" in item and isinstance(item["content"], str):
                    parts.append(item["content"])
                elif "text" in item and isinstance(item["text"], str):
                    parts.append(item["text"])
                else:
                    parts.append(json.dumps(item, ensure_ascii=False))
            else:
                parts.append(str(item))
        return "".join(parts)

    if isinstance(completion, dict):
        if "content" in completion and isinstance(completion["content"], str):
            return completion["content"]
        if "text" in completion and isinstance(completion["text"], str):
            return completion["text"]
        return json.dumps(completion, ensure_ascii=False)

    return str(completion)


def extract_json(completion) -> dict | None:
    """Extract JSON from model output, tolerating TRL structured completions."""
    text = normalize_completion_text(completion).strip()

    if text.startswith("```"):
        text = re.sub(r"^```[\w]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    return None


def reward_fn(completions, ground_truth, **kwargs) -> list[float]:
    scores = []
    for completion, gt_str in zip(completions, ground_truth):
        gt = json.loads(gt_str)
        parsed = extract_json(completion)

        if parsed is None:
            scores.append(0.01)
            continue

        if "issues" not in parsed and "output_data" in parsed:
            parsed = parsed["output_data"]

        score, _, _ = grade_task4(parsed, gt)
        scores.append(float(score))

    return scores


## Cell 5 — Baseline evaluation (before training)

In [5]:
import os
from pathlib import Path

import torch
from huggingface_hub import login, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
HF_CACHE_DIR = Path("/content/hf-cache")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print("Loaded HF token from Colab secrets.")
except Exception:
    print("No Colab HF token found. Continuing without explicit login.")

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required.")

compute_dtype = torch.float16

local_model_dir = snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=str(HF_CACHE_DIR),
    max_workers=2,
    ignore_patterns=["*.onnx", "*.tflite", "*.gguf", "*.md"],
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(local_model_dir, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    local_model_dir,
    local_files_only=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)


Loaded HF token from Colab secrets.


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [6]:
def run_eval(model, tokenizer, cases, label='eval'):
    """Run greedy inference on all cases and return mean score."""
    model.eval()
    scores_by_difficulty = {}
    all_scores = []

    for case in cases:
        prompt = case_to_prompt(case)
        text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        completion = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        gt_str = json.dumps(case['ground_truth'])
        score = reward_fn([completion], [gt_str])[0]

        difficulty = case.get('difficulty', 'unknown')
        scores_by_difficulty.setdefault(difficulty, []).append(score)
        all_scores.append(score)

        print(f"  [{label}] {case['case_id']} ({difficulty}): {score:.3f}")

    mean = sum(all_scores) / len(all_scores) if all_scores else 0.0
    print(f'\n[{label}] Overall: {mean:.4f}')
    for diff, s in sorted(scores_by_difficulty.items()):
        print(f'  {diff}: {sum(s)/len(s):.4f} ({len(s)} cases)')
    return mean, scores_by_difficulty, all_scores


# Evaluate on pharmaverse cases only (same split as our published baseline)
print('=== BASELINE (before training) ===')
baseline_mean, baseline_by_diff, baseline_scores = run_eval(
    base_model, tokenizer, pharmaverse_cases, label='baseline'
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== BASELINE (before training) ===
  [baseline] PV_T4_1 (easy): 0.800
  [baseline] PV_T4_2 (easy): 0.990
  [baseline] PV_T4_3 (medium): 0.990
  [baseline] PV_T4_4 (medium): 0.600
  [baseline] PV_T4_5 (medium): 0.250
  [baseline] PV_T4_6 (medium): 0.600
  [baseline] PV_T4_7 (hard): 0.880
  [baseline] PV_T4_8 (hard): 0.250
  [baseline] PV_T4_9 (hard): 0.933
  [baseline] PV_T4_10 (very_hard): 0.250

[baseline] Overall: 0.6543
  easy: 0.8950 (2 cases)
  hard: 0.6878 (3 cases)
  medium: 0.6100 (4 cases)
  very_hard: 0.2500 (1 cases)


## Cell 6 — Configure and run GRPO training

In [7]:
import trl, inspect
from trl import GRPOConfig

print(trl.__version__)
print(inspect.signature(GRPOConfig.__init__))


1.0.0
(self, output_dir: str | None = None, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: transformers.trainer_utils.IntervalStrategy | str = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, gradient_accumulation_steps: int = 1, eval_accumulation_steps: int | None = None, eval_delay: float = 0, torch_empty_cache_steps: int | None = None, learning_rate: float = 1e-06, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_ratio: float | None = None, warmup_steps: float = 0, log_level: str = 'passive', log_level_replica: str = 'warning', log_on_each_node: bool = True, logging_dir: str | None = None, logging_strategy: tr

In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer
import torch

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(base_model, lora_config)

# Important on Colab T4: keep trainable adapter params out of bf16
for param in peft_model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

peft_model.print_trainable_parameters()

grpo_config = GRPOConfig(
    output_dir="/content/grpo-task4",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    fp16=False,
    bf16=False,
    num_generations=2,
    max_completion_length=256,
    temperature=0.9,
    top_p=0.9,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=peft_model,
    reward_funcs=reward_fn,
    args=grpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print(f"Training on {len(dataset)} cases x {grpo_config.num_generations} generations x {grpo_config.num_train_epochs} epochs")
print("Starting GRPO training...")
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323
Training on 20 cases x 2 generations x 3 epochs
Starting GRPO training...


Step,Training Loss
5,-0.044400
10,0.033621
15,0.064181
20,0.053609
25,0.000000
30,0.075426


TrainOutput(global_step=30, training_loss=0.030406275391578676, metrics={'train_runtime': 1856.7464, 'train_samples_per_second': 0.032, 'train_steps_per_second': 0.016, 'total_flos': 0.0, 'train_loss': 0.030406275391578676})

## Cell 7 — Post-training evaluation

In [9]:
print('=== POST-TRAINING (after GRPO) ===')
trained_mean, trained_by_diff, trained_scores = run_eval(
    peft_model, tokenizer, pharmaverse_cases, label='trained'
)

=== POST-TRAINING (after GRPO) ===
  [trained] PV_T4_1 (easy): 0.800
  [trained] PV_T4_2 (easy): 0.990
  [trained] PV_T4_3 (medium): 0.990
  [trained] PV_T4_4 (medium): 0.600
  [trained] PV_T4_5 (medium): 0.250
  [trained] PV_T4_6 (medium): 0.600
  [trained] PV_T4_7 (hard): 0.880
  [trained] PV_T4_8 (hard): 0.250
  [trained] PV_T4_9 (hard): 0.933
  [trained] PV_T4_10 (very_hard): 0.250

[trained] Overall: 0.6543
  easy: 0.8950 (2 cases)
  hard: 0.6878 (3 cases)
  medium: 0.6100 (4 cases)
  very_hard: 0.2500 (1 cases)


## Cell 8 — Results comparison

In [10]:
print('\n' + '='*60)
print('RESULTS: Task 4 Cross-Domain Validation')
print('='*60)
print(f'  Model:    {MODEL_ID} + LoRA GRPO fine-tune')
print(f'  Dataset:  {len(pharmaverse_cases)} pharmaverse cases')
print(f'  Epochs:   {grpo_config.num_train_epochs}')
print()
print(f'  {"Metric":<25} {"Baseline":>10} {"Trained":>10} {"Delta":>10}')
print(f'  {"-"*55}')
print(f'  {"Overall mean":<25} {baseline_mean:>10.4f} {trained_mean:>10.4f} {trained_mean - baseline_mean:>+10.4f}')

all_diffs = sorted(set(list(baseline_by_diff.keys()) + list(trained_by_diff.keys())))
for diff in all_diffs:
    b = sum(baseline_by_diff.get(diff, [0])) / max(len(baseline_by_diff.get(diff, [1])), 1)
    t = sum(trained_by_diff.get(diff, [0])) / max(len(trained_by_diff.get(diff, [1])), 1)
    print(f'  {diff:<25} {b:>10.4f} {t:>10.4f} {t - b:>+10.4f}')

print('='*60)
improvement = trained_mean - baseline_mean
if improvement > 0:
    print(f'\nImprovement: +{improvement:.4f} ({improvement/baseline_mean*100:.1f}% relative)')
else:
    print(f'\nNo improvement this run. Try more epochs or higher learning rate.')


RESULTS: Task 4 Cross-Domain Validation
  Model:    Qwen/Qwen2.5-7B-Instruct + LoRA GRPO fine-tune
  Dataset:  10 pharmaverse cases
  Epochs:   3

  Metric                      Baseline    Trained      Delta
  -------------------------------------------------------
  Overall mean                  0.6543     0.6543    +0.0000
  easy                          0.8950     0.8950    +0.0000
  hard                          0.6878     0.6878    +0.0000
  medium                        0.6100     0.6100    +0.0000
  very_hard                     0.2500     0.2500    +0.0000

No improvement this run. Try more epochs or higher learning rate.


## Cell 9 — Save adapter and push to HuggingFace Hub

In [11]:
from huggingface_hub import HfApi

# Set your HuggingFace token here or via Kaggle secrets
HF_TOKEN = os.getenv('HF_TOKEN', '')  # add as Kaggle secret: Settings → Add-ons → Secrets
HF_REPO_ID = 'thisis-gp/clinical-data-grpo-task4'  # will be created if it doesn't exist

if HF_TOKEN:
    peft_model.save_pretrained('/kaggle/working/grpo-task4-adapter')
    tokenizer.save_pretrained('/kaggle/working/grpo-task4-adapter')

    peft_model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print(f'Adapter pushed to https://huggingface.co/{HF_REPO_ID}')
else:
    # Save locally if no HF token
    peft_model.save_pretrained('/kaggle/working/grpo-task4-adapter')
    tokenizer.save_pretrained('/kaggle/working/grpo-task4-adapter')
    print('Adapter saved to /kaggle/working/grpo-task4-adapter')
    print('Set HF_TOKEN in Kaggle Secrets to push to HuggingFace Hub.')

Adapter saved to /kaggle/working/grpo-task4-adapter
Set HF_TOKEN in Kaggle Secrets to push to HuggingFace Hub.


## Cell 10 — Generate README results snippet

In [12]:
# Copy-paste this into the README Training Results section
snippet = f"""## RL Training Results

Model: `{MODEL_ID}` fine-tuned with GRPO on Task 4 (cross-domain validation).

| Metric | Baseline (zero-shot) | After GRPO | Delta |
|--------|---------------------|------------|-------|
| Task 4 overall | {baseline_mean:.4f} | {trained_mean:.4f} | {trained_mean - baseline_mean:+.4f} |
"""
for diff in all_diffs:
    b = sum(baseline_by_diff.get(diff, [0])) / max(len(baseline_by_diff.get(diff, [1])), 1)
    t = sum(trained_by_diff.get(diff, [0])) / max(len(trained_by_diff.get(diff, [1])), 1)
    snippet += f'| Task 4 {diff} | {b:.4f} | {t:.4f} | {t-b:+.4f} |\n'

snippet += f"""
Training config: LoRA r=16, GRPO {grpo_config.num_generations} generations, \
{grpo_config.num_train_epochs} epochs, lr={grpo_config.learning_rate}
"""
print(snippet)

## RL Training Results

Model: `Qwen/Qwen2.5-7B-Instruct` fine-tuned with GRPO on Task 4 (cross-domain validation).

| Metric | Baseline (zero-shot) | After GRPO | Delta |
|--------|---------------------|------------|-------|
| Task 4 overall | 0.6543 | 0.6543 | +0.0000 |
| Task 4 easy | 0.8950 | 0.8950 | +0.0000 |
| Task 4 hard | 0.6878 | 0.6878 | +0.0000 |
| Task 4 medium | 0.6100 | 0.6100 | +0.0000 |
| Task 4 very_hard | 0.2500 | 0.2500 | +0.0000 |

Training config: LoRA r=16, GRPO 2 generations, 3 epochs, lr=5e-06

